<a href="https://colab.research.google.com/github/gunnsmart/science-skills/blob/arena%2F01a0b805-science-skills/Image_Upscaler_%26_QC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AI Image Upscaler + Image Quality Control Workbench

**Workflow:** เลือกโมเดล → เลือก settings → Upload ภาพ → UPSCALE → QC → Download

ออกแบบสำหรับ Google Colab โดยใช้ Colab Forms / file upload / display native เป็นหลัก ไม่มี Gradio, Streamlit, React หรือ web server.

> **Stock QC note:** Technical QC only — platform acceptance is not guaranteed.
>
> Notebook นี้ไม่ fake implementation: โมเดลที่ไม่มี checkpoint/official setup ที่ใช้งานได้กับ runtime ปัจจุบันจะแสดงเป็น `Unavailable on current runtime` พร้อมเหตุผล และจะไม่ถูกโหลดเข้า VRAM


**JPEG export:** Notebook now also creates `upscaled.jpg` and `upscaled_QC.jpg` by default. If source/result has alpha, JPEG copies are flattened on white because JPEG does not support transparency.


In [ ]:

# CELL 1 — Environment + GPU
#@title CELL 1 — Environment + GPU
import os, sys, gc, json, math, time, shutil, subprocess, platform, warnings, copy, hashlib
from pathlib import Path
from datetime import datetime, timezone

WORK_DIR = Path('/content/AI_Image_Upscaler_QC')
INPUT_DIR = WORK_DIR / 'inputs'
OUTPUT_DIR = WORK_DIR / 'outputs'
CACHE_DIR = WORK_DIR / 'cache'
for d in [INPUT_DIR, OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
AGGRESSIVE_CUDA_CLEANUP = True  # default; Cell 4 exposes this as a Colab Form control.

def run_cmd(cmd, check=False, quiet=False, cwd=None):
    print(f"$ {cmd}") if not quiet else None
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if (not quiet) and p.stdout:
        print(p.stdout[-4000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{p.stdout}")
    return p.returncode, p.stdout

def get_hardware_info():
    info = {
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'gpu': 'CPU only',
        'vram_total_gb': 0.0,
        'cuda_available': False,
        'cuda': None,
        'pytorch': None,
    }
    try:
        import torch
        info['pytorch'] = torch.__version__
        info['cuda_available'] = bool(torch.cuda.is_available())
        info['cuda'] = getattr(torch.version, 'cuda', None)
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            info['gpu'] = props.name
            info['vram_total_gb'] = round(props.total_memory / (1024**3), 2)
    except Exception as e:
        info['torch_error'] = str(e)
    return info

HARDWARE = get_hardware_info()
print(json.dumps(HARDWARE, indent=2))

def clear_vram(aggressive=None):
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            if aggressive is None:
                aggressive = bool(globals().get('AGGRESSIVE_CUDA_CLEANUP', False))
            if aggressive:
                torch.cuda.ipc_collect()
    except Exception:
        pass

clear_vram()


In [ ]:

# CELL 2 — Dependencies
#@title CELL 2 — Dependencies (core + QC; model-specific deps are lazy-installed only when selected)
import importlib.util, site

def pip_install(packages, extra_args=''):
    cmd = f"{sys.executable} -m pip install {extra_args} {packages}"
    code, out = run_cmd(cmd, quiet=False)
    if code != 0:
        raise RuntimeError(out)

CORE_PACKAGES = [
    'pillow>=10.0.0',
    'numpy',
    'matplotlib',
    'opencv-python-headless',
    'scikit-image',
    'pandas',
    'tqdm',
    'huggingface_hub',
    'safetensors',
    'requests',
]

# IQA-PyTorch provides MUSIQ, NIQE and BRISQUE in one toolbox.
# If this install fails, QC engine still computes Laplacian/FFT and reports metric errors explicitly.
QC_PACKAGES = ['pyiqa']

pip_install(' '.join(CORE_PACKAGES))
try:
    pip_install(' '.join(QC_PACKAGES))
    PYIQA_AVAILABLE = True
except Exception as e:
    PYIQA_AVAILABLE = False
    print('⚠️ pyiqa install failed. MUSIQ/NIQE/BRISQUE will be reported as unavailable, but Laplacian/FFT QC will still run.')
    print(str(e)[-2000:])

# Imports after install
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFont, ImageCms
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

print('Core dependencies ready.')


In [ ]:

# CELL 3 — Model Registry
#@title CELL 3 — Extensible Model Registry with verified official sources/checkpoint notes
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional

MODEL_REGISTRY = {
    # Implemented / lazy-loaded
    'Real-ESRGAN': {
        'family': 'GENERAL / GAN',
        'status': 'available',
        'implementation': 'python_api',
        'official_repo': 'https://github.com/xinntao/Real-ESRGAN',
        'weights': {
            'x4': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
            'x2': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
        },
        'native_scales': [2, 4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'Official xinntao Real-ESRGAN weights. 8x uses 4x native + safe resize, not native 8x.',
    },
    'Real-ESRGAN Anime': {
        'family': 'ANIME / ILLUSTRATION',
        'status': 'available',
        'implementation': 'python_api',
        'official_repo': 'https://github.com/xinntao/Real-ESRGAN',
        'weights': {
            'x4': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
        },
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'Official anime 6B x4 checkpoint. Uses internal SRVGG fallback on Colab Python 3.13.',
    },
    '4x-UltraSharp': {
        'family': 'GENERAL / GAN',
        'status': 'available_if_downloadable',
        'implementation': 'esrgan_rrdb_hf',
        'official_repo': 'https://huggingface.co/Kim2091/UltraSharp',
        'weights': {'x4_hf_repo': 'Kim2091/UltraSharp', 'filename': '4x-UltraSharp.pth'},
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'Community ESRGAN/RRDB checkpoint hosted on Hugging Face; notebook verifies availability before loading.',
    },
    'AuraSR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'available',
        'implementation': 'aura_sr',
        'official_repo': 'https://github.com/fal-ai/aura-sr',
        'weights': {'hf_model': 'fal-ai/AuraSR'},
        'native_scales': [4],
        'supports_tile': False,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': True,
        'notes': 'GAN/GigaGAN-derived SR package for AI-generated images. 4x only; overlapped mode reduces seams.',
    },

    # Registered but guarded: no fake implementation.
    'Real-CUGAN': {
        'family': 'ANIME / ILLUSTRATION',
        'status': 'available_if_binary_runs',
        'implementation': 'realcugan_ncnn',
        'official_repo': 'https://github.com/bilibili/ailab/tree/main/Real-CUGAN',
        'binary_repo': 'https://github.com/nihui/realcugan-ncnn-vulkan',
        'weights': {
            'ubuntu_zip': 'https://github.com/nihui/realcugan-ncnn-vulkan/releases/download/20220728/realcugan-ncnn-vulkan-20220728-ubuntu.zip',
        },
        'native_scales': [2,4],
        'supports_tile': True,
        'supports_precision': ['Auto'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'Real-CUGAN through nihui realcugan-ncnn-vulkan portable binary. Uses bundled models; may be unavailable if the Colab runtime cannot execute the binary.',
    },
    'SwinIR': {
        'family': 'TRANSFORMER',
        'status': 'available',
        'implementation': 'swinir_official',
        'official_repo': 'https://github.com/JingyunLiang/SwinIR',
        'weights': {
            'x4': 'https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth',
        },
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'Official SwinIR-M real-world SR x4 checkpoint; architecture is loaded lazily from the official repo.',
    },
    'HAT': {
        'family': 'TRANSFORMER',
        'status': 'available_if_downloadable',
        'implementation': 'spandrel_hf',
        'official_repo': 'https://github.com/XPixelGroup/HAT',
        'weights': {'hf_repo': 'jaideepsingh/upscale_models', 'hf_file': 'HAT/HAT-L_SRx4_ImageNet-pretrain.pth'},
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'HAT x4 via Spandrel auto-loader using a Hugging Face mirror of the checkpoint. Official project primarily distributes weights via Drive/Baidu.',
    },
    'Real-HAT': {
        'family': 'TRANSFORMER',
        'status': 'available_if_downloadable',
        'implementation': 'spandrel_hf',
        'official_repo': 'https://github.com/XPixelGroup/HAT',
        'weights': {'hf_repo': 'jaideepsingh/upscale_models', 'hf_file': 'HAT/Real_HAT_GAN_SRx4.pth'},
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'Real-HAT x4 via Spandrel auto-loader using a Hugging Face mirror; avoids BasicSR runtime dependencies.',
    },
    'DAT': {
        'family': 'TRANSFORMER',
        'status': 'available_if_downloadable',
        'implementation': 'spandrel_hf',
        'official_repo': 'https://github.com/zhengchen1999/DAT',
        'weights': {'hf_repo': 'w-e-w/DAT', 'hf_file': 'experiments/pretrained_models/DAT/DAT_x4.pth'},
        'native_scales': [4],
        'supports_tile': True,
        'supports_precision': ['Auto','FP16','FP32'],
        'supports_face_restoration': ['None','GFPGAN','CodeFormer'],
        'supports_generative_detail': False,
        'notes': 'DAT x4 via Spandrel auto-loader using a Hugging Face mirror of the official checkpoint.',
    },
    'SUPIR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/Fanghua-Yu/SUPIR',
        'native_scales': [1,2,4],
        'reason': 'Requires SDXL/SUPIR checkpoints and high VRAM. Current turnkey Colab path would need user-managed model files, so disabled.',
    },
    'OSEDiff': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/cswry/OSEDiff',
        'native_scales': [4],
        'reason': 'Diffusion dependencies/checkpoints are not safely packaged for this single-notebook runtime yet.',
    },
    'TSD-SR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/Microtreei/TSD-SR',
        'native_scales': [4],
        'reason': 'Requires SD3 model path plus LoRA/prompt embeddings. Disabled because it would require manual checkpoint paths.',
    },
    'InvSR': {
        'family': 'GENERATIVE / DIFFUSION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/zsyOAOA/InvSR',
        'native_scales': [4],
        'reason': 'Requires SD-Turbo plus InvSR checkpoint; not included to avoid hidden downloads and VRAM failures.',
    },
    'Anime4K': {
        'family': 'ANIME / ILLUSTRATION',
        'status': 'unavailable',
        'official_repo': 'https://github.com/bloc97/Anime4K',
        'native_scales': [2,4,8],
        'reason': 'Shader/libplacebo/Vulkan workflow is not a native Colab image notebook path. Not loaded.',
    },
    'CodeFormer': {
        'family': 'FACE RESTORATION',
        'status': 'available_as_face_restoration',
        'implementation': 'spandrel_face_restoration',
        'official_repo': 'https://github.com/sczhou/CodeFormer',
        'weights': {'url': 'https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth'},
        'native_scales': [1],
        'reason': 'Available as optional post-upscale face restoration, not a full-image SR model.',
    },
    'GFPGAN': {
        'family': 'FACE RESTORATION',
        'status': 'available_as_face_restoration',
        'implementation': 'spandrel_face_restoration',
        'official_repo': 'https://github.com/TencentARC/GFPGAN',
        'weights': {'url': 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth'},
        'native_scales': [1],
        'reason': 'Available as optional post-upscale face restoration, not a full-image SR model.',
    },
}

AVAILABLE_MAIN_MODELS = [k for k,v in MODEL_REGISTRY.items() if v.get('implementation') in ['python_api','esrgan_rrdb_hf','aura_sr','swinir_official','realcugan_ncnn','spandrel_hf']]

# Alpha handling cache/policy. Initialized here so all later cells can reuse it.
ALPHA_CACHE = {}
ALPHA_POLICY = {
    'model_input': 'RGB/BGR only',
    'alpha_strategy': 'cache/source alpha separately, resize after SR, then merge for PNG-capable outputs',
    'jpeg_alpha_strategy': 'flatten on white',
}

def show_table(rows, title=None, max_colwidth=72):
    """Plain-text tables avoid Colab's verbose dataframe HTML/CSS output.

    This function imports pandas lazily so CELL 3 is still usable even if a user
    runs it out of order or CELL 2 did not finish cleanly.
    """
    if title:
        print(f'\n{title}')
        print('=' * len(title))
    try:
        import pandas as _pd
        df = _pd.DataFrame(rows)
        with _pd.option_context('display.max_colwidth', max_colwidth, 'display.width', 220, 'display.max_columns', 20):
            print(df.to_string(index=False))
    except Exception as e:
        print(f'⚠️ Table formatter fallback because pandas is unavailable: {e}')
        if not rows:
            print('(empty)')
            return
        headers = list(rows[0].keys())
        widths = {h: min(max(len(str(h)), *(len(str(r.get(h, ''))) for r in rows)), max_colwidth) for h in headers}
        print(' | '.join(str(h)[:widths[h]].ljust(widths[h]) for h in headers))
        print('-+-'.join('-' * widths[h] for h in headers))
        for r in rows:
            print(' | '.join(str(r.get(h, ''))[:widths[h]].ljust(widths[h]) for h in headers))

registry_rows = []
for name, meta in MODEL_REGISTRY.items():
    registry_rows.append({
        'Model': name,
        'Family': meta.get('family'),
        'Status': meta.get('status'),
        'Native scales': ','.join(map(str, meta.get('native_scales', []))),
        'Reason/Notes': meta.get('reason', meta.get('notes','')),
    })
show_table(registry_rows, title='Model Registry')
print('\nAvailable main models:', ', '.join(AVAILABLE_MAIN_MODELS))


In [ ]:

# CELL 4 — Main User Interface / Controls
#@title CELL 4 — Main User Interface / Controls
MODEL = "Real-ESRGAN" #@param ["Real-ESRGAN", "Real-ESRGAN Anime", "4x-UltraSharp", "AuraSR", "Real-CUGAN", "SwinIR", "HAT", "Real-HAT", "DAT", "SUPIR", "OSEDiff", "TSD-SR", "InvSR", "Anime4K", "CodeFormer", "GFPGAN"]
SCALE = "4x" #@param ["2x", "4x", "8x"]
QUALITY_PRESET = "Balanced" #@param ["Fast", "Balanced", "High Quality"]
TILE = "Auto" #@param ["Auto", "256", "512", "768", "1024"]
PRECISION = "Auto" #@param ["Auto", "FP16", "BF16", "FP32"]
FACE_RESTORATION = "None" #@param ["None", "CodeFormer", "GFPGAN"]
GENERATIVE_DETAIL = "OFF" #@param ["OFF", "ON"]
PRESERVE_ORIGINAL = 0 #@param {type:"slider", min:0, max:100, step:5}
STOCK_QC = False #@param {type:"boolean"}
COMPARE_MODE = False #@param {type:"boolean"}
COMPARE_MODELS = "Real-ESRGAN, 4x-UltraSharp, AuraSR" #@param {type:"string"}
KEEP_QC_CACHE = False #@param {type:"boolean"}
EXPORT_JPEG = True #@param {type:"boolean"}
JPEG_QUALITY = 95 #@param {type:"slider", min:70, max:100, step:1}
AGGRESSIVE_CUDA_CLEANUP = True #@param {type:"boolean"}

MODEL_SETTINGS = {
    'model': MODEL,
    'scale': int(SCALE.replace('x','')),
    'quality_preset': QUALITY_PRESET,
    'tile': TILE,
    'precision': PRECISION,
    'face_restoration': FACE_RESTORATION,
    'generative_detail': GENERATIVE_DETAIL == 'ON',
    'preserve_original_percent': int(PRESERVE_ORIGINAL),
    'stock_qc': bool(STOCK_QC),
    'compare_mode': bool(COMPARE_MODE),
    'compare_models': [m.strip() for m in COMPARE_MODELS.split(',') if m.strip()],
    'keep_qc_cache': bool(KEEP_QC_CACHE),
    'export_jpeg': bool(EXPORT_JPEG),
    'jpeg_quality': int(JPEG_QUALITY),
    'aggressive_cuda_cleanup': bool(AGGRESSIVE_CUDA_CLEANUP),
}

def normalize_settings(settings):
    model = settings['model']
    meta = MODEL_REGISTRY.get(model, {})
    normalized = copy.deepcopy(settings)
    warnings_list = []

    if meta.get('status') == 'unavailable':
        raise RuntimeError(f"{model}: Unavailable on current runtime — {meta.get('reason')}")
    if not meta.get('implementation'):
        raise RuntimeError(f"{model}: Unavailable as a main image upscaler in this notebook — {meta.get('reason', 'No full-image SR implementation is registered.')}")

    if normalized['scale'] not in meta.get('native_scales', []):
        if model in ['Real-ESRGAN', '4x-UltraSharp'] and normalized['scale'] in [2,4,8]:
            warnings_list.append(f"Requested {normalized['scale']}x is not native for {model}; output will use model native scale plus high-quality resize if needed.")
        else:
            native = meta.get('native_scales')
            raise RuntimeError(f"{model} supports native scale(s) {native}, not {normalized['scale']}x.")

    if not meta.get('supports_tile', False):
        normalized['tile'] = 'Not supported'
    if normalized['precision'] == 'BF16' and 'BF16' not in meta.get('supports_precision', []):
        warnings_list.append(f"{model} does not support BF16 in this notebook; using Auto.")
        normalized['precision'] = 'Auto'
    if normalized['face_restoration'] not in meta.get('supports_face_restoration', ['None']):
        warnings_list.append(f"Face restoration {normalized['face_restoration']} is not supported with {model}; using None.")
        normalized['face_restoration'] = 'None'
    if normalized['generative_detail'] and not meta.get('supports_generative_detail', False):
        warnings_list.append(f"Generative Detail is not supported by {model}; ignored.")
        normalized['generative_detail'] = False
    return normalized, warnings_list

try:
    preview_candidates = MODEL_SETTINGS['compare_models'] if MODEL_SETTINGS.get('compare_mode') else [MODEL]
    preview_errors = []
    NORMALIZED_SETTINGS, SETTING_WARNINGS, PREVIEW_MODEL = None, [], None
    for candidate in preview_candidates:
        try:
            candidate_settings = copy.deepcopy(MODEL_SETTINGS)
            candidate_settings['model'] = candidate
            NORMALIZED_SETTINGS, SETTING_WARNINGS = normalize_settings(candidate_settings)
            PREVIEW_MODEL = candidate
            break
        except Exception as candidate_error:
            preview_errors.append(f'{candidate}: {candidate_error}')
    if PREVIEW_MODEL is None:
        raise RuntimeError('No runnable model found in the current selection. ' + ' | '.join(preview_errors))

    title = 'Compare Mode first runnable model capability' if MODEL_SETTINGS.get('compare_mode') else 'Selected model capability'
    display(Markdown(f'### {title}'))
    meta = MODEL_REGISTRY.get(PREVIEW_MODEL, {})
    show_table([{
        'Model': PREVIEW_MODEL,
        'Status': meta.get('status'),
        'Native scales': ','.join(map(str, meta.get('native_scales', []))),
        'Tile': meta.get('supports_tile'),
        'Precision': ','.join(meta.get('supports_precision', [])),
        'Face restoration': ','.join(meta.get('supports_face_restoration', ['None'])),
        'Generative detail': meta.get('supports_generative_detail', False),
    }], title='Model Capability Preview')
    if MODEL_SETTINGS.get('compare_mode'):
        print('Compare Mode models:', ', '.join(MODEL_SETTINGS['compare_models']))
        if preview_errors:
            print('Skipped preview candidates:')
            for msg in preview_errors:
                print('⚠️', msg)
    if SETTING_WARNINGS:
        display(Markdown('### Setting warnings'))
        for w in SETTING_WARNINGS: print('⚠️', w)
except Exception as e:
    NORMALIZED_SETTINGS = None
    SETTING_WARNINGS = [str(e)]
    print('❌', e)

if STOCK_QC:
    print('Stock QC enabled: Technical QC only — platform acceptance is not guaranteed.')


In [ ]:

# CELL 5 — Upload + Upscale Engine
#@title CELL 5 — Upload Image + UPSCALE
RUN_UPSCALE = True #@param {type:"boolean"}

from google.colab import files
import urllib.request
from io import BytesIO
import zipfile

SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}
MODEL_CACHE = {}
EXPECTED_CHECKPOINT_BYTES = {
    'RealESRGAN_x4plus.pth': 67040989,
    'RealESRGAN_x2plus.pth': 67061725,
    'RealESRGAN_x4plus_anime_6B.pth': 17938799,
    # Verified from GitHub release API after curl -I confirmed the v1.3.0 release endpoint.
    'GFPGANv1.3.pth': 348632874,
    'codeformer.pth': 376637898,
}
# Optional integrity map. Fill only with hashes that have been independently verified.
CHECKPOINT_SHA256 = {}
EXPECTED_BINARY_BYTES = {
    'realcugan-ncnn-vulkan-20220728-ubuntu.zip': 46704187,
}

# ---------- dependency isolation / lazy installers ----------
def ensure_realesrgan_stack():
    if importlib.util.find_spec('realesrgan') and importlib.util.find_spec('gfpgan') and importlib.util.find_spec('basicsr'):
        patch_torchvision_functional_tensor()
        return
    print('Installing Real-ESRGAN/GFPGAN stack for selected model only...')
    print('Using a Colab-safe installer: keep setuptools compatible with torch and disable BasicSR build isolation.')
    # Colab Python 3.13 / torch 2.11 currently requires setuptools<82. Do NOT upgrade setuptools blindly.
    # Build isolation can pull a too-new setuptools and make BasicSR fail with empty build output.
    pip_install('"setuptools<82" "wheel<0.49"')
    pip_install('addict future lmdb pyyaml requests scipy tb-nightly yapf')
    # Install BasicSR without dependency/build isolation, then install wrappers without allowing pip to replace BasicSR.
    basicsr_sources = [
        'git+https://github.com/XPixelGroup/BasicSR@8d56e3a045f9fb3e1d8872f92ee4a4f07f886b0a',
        'basicsr==1.4.2',
    ]
    last_error = None
    for src in basicsr_sources:
        try:
            pip_install(src, extra_args='--no-build-isolation --no-deps')
            last_error = None
            break
        except Exception as e:
            last_error = e
            print(f'⚠️ BasicSR install attempt failed for {src}: {str(e)[-1200:]}')
    if last_error is not None:
        raise RuntimeError(
            'BasicSR could not be installed in this runtime. This usually means the current Colab Python/PyTorch image is incompatible. '
            'Try Runtime → Disconnect and delete runtime, then run again; if it still fails, use AuraSR or wait for a Colab image with Python <=3.12.\n'
            f'Last BasicSR error: {last_error}'
        )
    pip_install('facexlib', extra_args='--no-build-isolation')
    pip_install('gfpgan realesrgan', extra_args='--no-build-isolation --no-deps')
    patch_torchvision_functional_tensor()


def patch_torchvision_functional_tensor():
    # Some BasicSR releases import torchvision.transforms.functional_tensor, removed in newer torchvision.
    try:
        import sys, types
        import torchvision.transforms.functional as F
        mod = types.ModuleType('torchvision.transforms.functional_tensor')
        for name in dir(F):
            setattr(mod, name, getattr(F, name))
        sys.modules['torchvision.transforms.functional_tensor'] = mod
    except Exception as e:
        print('Patch warning:', e)


def ensure_aura_sr():
    if importlib.util.find_spec('aura_sr'):
        return
    print('Installing AuraSR for selected model only...')
    pip_install('aura-sr')

# ---------- image validation ----------
def validate_uploaded_image(path: Path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    w, h = img.size
    mode = img.mode
    has_alpha = mode in ('RGBA','LA') or ('transparency' in img.info)
    size_mb = path.stat().st_size / (1024*1024)
    info = {
        'filename': path.name,
        'path': str(path),
        'resolution': [w, h],
        'aspect_ratio': round(w / h, 6) if h else None,
        'color_mode': mode,
        'alpha_channel': bool(has_alpha),
        'file_size_mb': round(size_mb, 3),
    }
    if path.suffix.lower() not in SUPPORTED_EXTS:
        raise ValueError(f'Unsupported file type: {path.suffix}. Use PNG, JPG/JPEG, or WEBP.')
    if mode == 'CMYK':
        print('⚠️ CMYK input detected. It will be converted to RGB for model inference; color may shift slightly. Consider uploading sRGB PNG/JPEG for best consistency.')
    if w < 8 or h < 8:
        raise ValueError('Image is too small for upscaling/QC.')
    if w < 64 or h < 64:
        print('⚠️ Very small image detected (<64px on one side). SR and IQA metrics may be unstable.')
    if size_mb > 80:
        print('⚠️ Large file; Colab memory/VRAM may be stressed. Tiling will be used when supported.')
    return img, info


def upload_one_image():
    print('Upload one PNG/JPG/JPEG/WEBP image...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No image uploaded.')
    if len(uploaded) > 1:
        print(f'⚠️ Multiple files uploaded ({len(uploaded)}). This notebook run will process the first file only. Use Compare Mode for multiple models on one image, or rerun this cell for another image.')
    name = next(iter(uploaded.keys()))
    src = INPUT_DIR / name
    with open(src, 'wb') as f:
        f.write(uploaded[name])
    img, info = validate_uploaded_image(src)
    normalized_path = INPUT_DIR / 'input_normalized.png'
    # Keep alpha when present; otherwise RGB.
    if info['alpha_channel']:
        img.convert('RGBA').save(normalized_path)
    else:
        img.convert('RGB').save(normalized_path)
    info['normalized_path'] = str(normalized_path)
    print('Input image info:')
    print(json.dumps(info, indent=2))
    return normalized_path, info

# ---------- automatic VRAM/precision/tile ----------
def effective_tile(tile_setting, supports_tile=True):
    if not supports_tile:
        return 0
    if tile_setting != 'Auto':
        return int(tile_setting)
    vram = HARDWARE.get('vram_total_gb', 0)
    if vram == 0:
        return 256
    if vram < 8:
        return 256
    if vram < 16:
        return 512
    return 0  # Real-ESRGAN uses 0 for no tiling


def effective_precision(precision):
    import torch
    if precision == 'Auto':
        return 'FP16' if torch.cuda.is_available() else 'FP32'
    if precision == 'FP16' and not torch.cuda.is_available():
        print('⚠️ FP16 requested but CUDA unavailable; using FP32.')
        return 'FP32'
    if precision == 'BF16':
        # Only used by models that support it; otherwise normalized before.
        return 'BF16'
    return precision

# ---------- model runners ----------
def checkpoint_size_ok(path: Path, expected_bytes=None, tolerance=None, min_bytes=1024*1024):
    if not path.exists():
        return False, 'missing'
    size = path.stat().st_size
    if expected_bytes:
        # Small checkpoints stay strict; large checkpoints can vary slightly by mirror/CDN metadata.
        tolerance = 0.10 if tolerance is None and expected_bytes >= 100 * 1024 * 1024 else (0.05 if tolerance is None else tolerance)
        lower = int(expected_bytes * (1 - tolerance))
        upper = int(expected_bytes * (1 + tolerance))
        ok = lower <= size <= upper
        return ok, f'{size} bytes; expected {expected_bytes} bytes ±{int(tolerance*100)}%'
    ok = size >= min_bytes
    return ok, f'{size} bytes; minimum {min_bytes} bytes'


def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def download_url(url, dst: Path, min_bytes=1024*1024, retries=3, expected_bytes=None):
    dst.parent.mkdir(parents=True, exist_ok=True)
    expected_bytes = expected_bytes or EXPECTED_CHECKPOINT_BYTES.get(dst.name)
    ok, reason = checkpoint_size_ok(dst, expected_bytes=expected_bytes, min_bytes=min_bytes)
    if ok:
        expected_sha = CHECKPOINT_SHA256.get(dst.name)
        if expected_sha:
            actual_sha = sha256_file(dst)
            if actual_sha.lower() != expected_sha.lower():
                print(f'Existing file failed SHA256 check; re-downloading: {dst}')
                dst.unlink(missing_ok=True)
            else:
                return dst
        else:
            return dst
    if dst.exists():
        print(f'Existing file failed size check ({reason}); re-downloading: {dst}')
        dst.unlink(missing_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            print(f'Downloading ({attempt}/{retries}) {url} -> {dst}')
            try:
                import requests
                with requests.get(url, stream=True, timeout=(10, 60), headers={'User-Agent': 'Mozilla/5.0'}) as r:
                    r.raise_for_status()
                    with open(dst, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)
            except Exception:
                # Fallback for environments where requests streaming is unavailable.
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=60) as response, open(dst, 'wb') as f:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        f.write(chunk)
            ok, reason = checkpoint_size_ok(dst, expected_bytes=expected_bytes, min_bytes=min_bytes)
            if ok:
                expected_sha = CHECKPOINT_SHA256.get(dst.name)
                if expected_sha:
                    actual_sha = sha256_file(dst)
                    if actual_sha.lower() != expected_sha.lower():
                        raise RuntimeError(f'Downloaded checkpoint failed SHA256 check: {actual_sha}')
                return dst
            raise RuntimeError(f'Downloaded checkpoint failed size check: {reason}')
        except Exception as e:
            last_error = e
            dst.unlink(missing_ok=True)
            print(f'⚠️ Download failed: {e}')
            time.sleep(2 * attempt)
    raise RuntimeError(f'Could not download checkpoint after {retries} attempts: {url}\nLast error: {last_error}')



def _has_alpha(pil_img: Image.Image):
    """Detect true alpha channels only; keep RGB transparency-key handling out of SR alpha paths."""
    if pil_img.mode in ('RGBA', 'LA', 'PA'):
        return True
    # Palette transparency may be an int, bytes, or list. It becomes a true alpha plane after RGBA conversion.
    if pil_img.mode == 'P' and 'transparency' in pil_img.info:
        return True
    return False


def convert_to_srgb_rgb(pil_img: Image.Image):
    """Convert PIL image to sRGB RGB, using embedded ICC for CMYK when available."""
    if pil_img.mode == 'CMYK':
        icc = pil_img.info.get('icc_profile')
        if icc:
            try:
                src_profile = ImageCms.ImageCmsProfile(BytesIO(icc))
                dst_profile = ImageCms.createProfile('sRGB')
                return ImageCms.profileToProfile(pil_img, src_profile, dst_profile, outputMode='RGB')
            except Exception as e:
                print(f'⚠️ CMYK ICC conversion failed; falling back to PIL convert("RGB"): {e}')
        else:
            print('⚠️ CMYK image has no embedded ICC profile; falling back to PIL convert("RGB").')
    return pil_img.convert('RGB')


def load_cv2_image_exif_safe(path: Path, keep_alpha=True):
    """Apply EXIF orientation, feed only RGB/BGR to SR, and return alpha separately.

    RRDB/SRVGG models are 3-channel models. Passing BGRA can cause shape/model errors or
    inconsistent alpha processing, so alpha is cached/resized separately after SR.
    """
    pil = ImageOps.exif_transpose(Image.open(path))
    alpha = None
    if pil.mode == 'CMYK':
        print('⚠️ CMYK image converted to sRGB/RGB for model inference.')
    if keep_alpha and _has_alpha(pil):
        rgba = pil.convert('RGBA')  # always convert before extracting alpha; handles palette transparency correctly.
        alpha = np.asarray(rgba.getchannel('A'), dtype=np.uint8)
        rgb = convert_to_srgb_rgb(rgba)
        globals().setdefault('ALPHA_CACHE', {})[str(path)] = alpha
    else:
        rgb = convert_to_srgb_rgb(pil)
    rgb_arr = np.asarray(rgb, dtype=np.uint8)
    return cv2.cvtColor(rgb_arr, cv2.COLOR_RGB2BGR), alpha


def alpha_resize_interpolation(alpha):
    # Binary masks should stay binary; soft alpha needs high-quality interpolation.
    unique = np.unique(alpha)
    if unique.size <= 2 and set(unique.tolist()).issubset({0, 255}):
        return cv2.INTER_NEAREST, 'binary-alpha-nearest'
    return cv2.INTER_LINEAR, 'soft-alpha-linear'


def merge_resized_alpha_if_needed(bgr_or_bgra, alpha):
    if alpha is None:
        return bgr_or_bgra
    h, w = bgr_or_bgra.shape[:2]
    interpolation, alpha_mode = alpha_resize_interpolation(alpha)
    alpha_up = cv2.resize(alpha, (w, h), interpolation=interpolation)
    print(f'Alpha channel restored using {alpha_mode}.')
    if bgr_or_bgra.ndim == 3 and bgr_or_bgra.shape[2] == 4:
        out = bgr_or_bgra.copy()
        out[:, :, 3] = alpha_up
        return out
    return cv2.merge([bgr_or_bgra[:, :, 0], bgr_or_bgra[:, :, 1], bgr_or_bgra[:, :, 2], alpha_up]) if bgr_or_bgra.shape[2] == 3 else bgr_or_bgra


def flatten_bgra_on_white(img):
    if img.ndim != 3 or img.shape[2] != 4:
        return img
    bgr = img[:, :, :3].astype(np.float32)
    alpha = (img[:, :, 3:4].astype(np.float32) / 255.0)
    white = np.full_like(bgr, 255, dtype=np.float32)
    return np.clip(bgr * alpha + white * (1.0 - alpha), 0, 255).astype(np.uint8)


def save_cv2_image(path: Path, img):
    params = []
    suffix = path.suffix.lower()
    if suffix in ['.jpg', '.jpeg']:
        if img.ndim == 3 and img.shape[2] == 4:
            print('JPEG does not support alpha; flattening on white background before save.')
            img = flatten_bgra_on_white(img)
        params.extend([cv2.IMWRITE_JPEG_QUALITY, 95])
        if hasattr(cv2, 'IMWRITE_JPEG_SAMPLING_FACTOR') and hasattr(cv2, 'IMWRITE_JPEG_SAMPLING_FACTOR_444'):
            params.extend([cv2.IMWRITE_JPEG_SAMPLING_FACTOR, cv2.IMWRITE_JPEG_SAMPLING_FACTOR_444])
    ok = cv2.imwrite(str(path), img, params)
    if not ok:
        raise RuntimeError(f'OpenCV could not write output image: {path}')


def safe_model_slug(name: str, used=None):
    slug = ''.join(c if c.isalnum() else '_' for c in name).strip('_')
    while '__' in slug:
        slug = slug.replace('__', '_')
    slug = slug or 'model'
    if used is None:
        return slug
    count = used.get(slug, 0) + 1
    used[slug] = count
    return slug if count == 1 else f'{slug}_{count}'


def unload_between_models(label='model', aggressive=False):
    print(f'Cleaning VRAM after {label}...')
    clear_vram(aggressive=aggressive)



# ---------- BasicSR-free Real-ESRGAN fallback engine ----------
def pixel_unshuffle_torch(x, scale):
    b, c, h, w = x.size()
    assert h % scale == 0 and w % scale == 0, f'Input size must be divisible by {scale}'
    out_channel = c * (scale ** 2)
    out_h = h // scale
    out_w = w // scale
    x_view = x.view(b, c, out_h, scale, out_w, scale)
    return x_view.permute(0, 1, 3, 5, 2, 4).reshape(b, out_channel, out_h, out_w)


def build_internal_rrdbnet(scale):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class ResidualDenseBlock(nn.Module):
        def __init__(self, num_feat=64, num_grow_ch=32):
            super().__init__()
            self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
            self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
            self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
            self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
            self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)
            self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        def forward(self, x):
            x1 = self.lrelu(self.conv1(x))
            x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
            x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
            x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
            x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
            return x5 * 0.2 + x

    class RRDB(nn.Module):
        def __init__(self, num_feat, num_grow_ch=32):
            super().__init__()
            self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
            self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
            self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)
        def forward(self, x):
            out = self.rdb1(x)
            out = self.rdb2(out)
            out = self.rdb3(out)
            return out * 0.2 + x

    class RRDBNet(nn.Module):
        def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4):
            super().__init__()
            self.scale = scale
            first_in = num_in_ch * 4 if scale == 2 else num_in_ch
            self.conv_first = nn.Conv2d(first_in, num_feat, 3, 1, 1)
            self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
            self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
            self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
            self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
            self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
            self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
            self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        def forward(self, x):
            if self.scale == 2:
                # x2plus is trained with pixel-unshuffle x2 then two x2 upsampling stages.
                if x.shape[-2] % 2 or x.shape[-1] % 2:
                    x = F.pad(x, (0, x.shape[-1] % 2, 0, x.shape[-2] % 2), mode='reflect')
                feat = pixel_unshuffle_torch(x, 2)
            else:
                feat = x
            feat = self.conv_first(feat)
            body_feat = self.conv_body(self.body(feat))
            feat = feat + body_feat
            feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode='nearest')))
            feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode='nearest')))
            out = self.conv_last(self.lrelu(self.conv_hr(feat)))
            return out
    return RRDBNet(scale=scale)


def load_internal_state_dict(model, weight_path, device):
    import torch
    loadnet = torch.load(str(weight_path), map_location=device)
    if isinstance(loadnet, dict):
        if 'params_ema' in loadnet:
            loadnet = loadnet['params_ema']
        elif 'params' in loadnet:
            loadnet = loadnet['params']
        elif 'state_dict' in loadnet:
            loadnet = loadnet['state_dict']
    clean = {k.replace('module.', ''): v for k, v in loadnet.items()}
    model.load_state_dict(clean, strict=True)
    model.eval().to(device)
    return model


def internal_model_forward_tiled(model, tensor, scale, tile=0, tile_pad=10):
    import torch
    import torch.nn.functional as F
    _, _, h, w = tensor.shape
    # Pad x2plus input to even dimensions before splitting; crop final back after SR.
    pad_h = h % 2 if scale == 2 else 0
    pad_w = w % 2 if scale == 2 else 0
    if pad_h or pad_w:
        tensor = F.pad(tensor, (0, pad_w, 0, pad_h), mode='reflect')
        h, w = tensor.shape[-2:]
    if not tile or tile <= 0:
        with torch.no_grad():
            out = model(tensor)
        return out[:, :, : (h - pad_h) * scale, : (w - pad_w) * scale]
    out = torch.zeros((1, 3, h * scale, w * scale), device=tensor.device, dtype=tensor.dtype)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            y0, x0 = y, x
            y1, x1 = min(y + tile, h), min(x + tile, w)
            py0, px0 = max(y0 - tile_pad, 0), max(x0 - tile_pad, 0)
            py1, px1 = min(y1 + tile_pad, h), min(x1 + tile_pad, w)
            inp = tensor[:, :, py0:py1, px0:px1]
            with torch.no_grad():
                pred = model(inp)
            oy0, ox0, oy1, ox1 = y0 * scale, x0 * scale, y1 * scale, x1 * scale
            cy0, cx0 = (y0 - py0) * scale, (x0 - px0) * scale
            cy1, cx1 = cy0 + (y1 - y0) * scale, cx0 + (x1 - x0) * scale
            out[:, :, oy0:oy1, ox0:ox1] = pred[:, :, cy0:cy1, cx0:cx1]
    return out[:, :, : (h - pad_h) * scale, : (w - pad_w) * scale]


def run_internal_rrdb_upscale(input_path: Path, output_path: Path, settings: dict, ultrasharp=False):
    import torch
    from huggingface_hub import hf_hub_download
    clear_vram()
    scale_req = int(settings['scale'])
    precision = effective_precision(settings['precision'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tile = effective_tile(settings['tile'], True)
    if ultrasharp:
        native_scale = 4
        weight_path = hf_hub_download(repo_id='Kim2091/UltraSharp', filename='4x-UltraSharp.pth', cache_dir=str(CACHE_DIR/'hf'))
    else:
        native_scale = 2 if scale_req == 2 else 4
        filename = 'RealESRGAN_x2plus.pth' if native_scale == 2 else 'RealESRGAN_x4plus.pth'
        key = 'x2' if native_scale == 2 else 'x4'
        weight_path = download_url(MODEL_REGISTRY['Real-ESRGAN']['weights'][key], CACHE_DIR / 'weights' / filename)
    model = build_internal_rrdbnet(native_scale)
    model = load_internal_state_dict(model, weight_path, device)
    if precision == 'FP16' and torch.cuda.is_available():
        model = model.half()
    img, alpha = load_cv2_image_exif_safe(input_path, keep_alpha=True)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(np.transpose(rgb, (2, 0, 1))).float().unsqueeze(0) / 255.0
    tensor = tensor.to(device)
    if precision == 'FP16' and torch.cuda.is_available():
        tensor = tensor.half()
    try:
        out = internal_model_forward_tiled(model, tensor, native_scale, tile=tile, tile_pad=10)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower() and (not tile or tile > 256):
            print('⚠️ CUDA OOM in internal RRDB engine; retrying with tile=256...')
            clear_vram(aggressive=True)
            out = internal_model_forward_tiled(model, tensor, native_scale, tile=256, tile_pad=10)
            tile = 256
        elif precision == 'FP16':
            print('⚠️ FP16 internal RRDB failed; retrying FP32...')
            retry_settings = copy.deepcopy(settings)
            retry_settings['precision'] = 'FP32'
            return run_internal_rrdb_upscale(input_path, output_path, retry_settings, ultrasharp=ultrasharp)
        else:
            raise
    out = out.float().clamp_(0, 1).detach().cpu().numpy()[0]
    out_rgb = np.transpose(out, (1, 2, 0))
    out_bgr = cv2.cvtColor((out_rgb * 255.0).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
    if int(settings['scale']) != native_scale:
        target_w = int(Image.open(input_path).size[0] * int(settings['scale']))
        target_h = int(Image.open(input_path).size[1] * int(settings['scale']))
        out_bgr = cv2.resize(out_bgr, (target_w, target_h), interpolation=cv2.INTER_LANCZOS4)
    out_bgr = merge_resized_alpha_if_needed(out_bgr, alpha)
    save_cv2_image(output_path, out_bgr)
    del model, tensor
    clear_vram()
    return {'effective_precision': precision, 'tile': tile, 'native_scale': native_scale, 'engine': 'internal_rrdb_no_basicsr'}



def build_internal_srvggnet(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4, act_type='prelu'):
    import torch.nn as nn
    import torch.nn.functional as F

    class SRVGGNetCompact(nn.Module):
        def __init__(self):
            super().__init__()
            self.upscale = upscale
            if act_type == 'prelu':
                activation = lambda: nn.PReLU(num_parameters=num_feat)
            elif act_type == 'relu':
                activation = lambda: nn.ReLU(inplace=True)
            else:
                activation = lambda: nn.LeakyReLU(negative_slope=0.1, inplace=True)
            body = []
            body += [nn.Conv2d(num_in_ch, num_feat, 3, 1, 1), activation()]
            for _ in range(num_conv):
                body += [nn.Conv2d(num_feat, num_feat, 3, 1, 1), activation()]
            body += [nn.Conv2d(num_feat, num_out_ch * (upscale ** 2), 3, 1, 1), nn.PixelShuffle(upscale)]
            self.body = nn.Sequential(*body)
        def forward(self, x):
            out = self.body(x)
            base = F.interpolate(x, scale_factor=self.upscale, mode='nearest')
            return out + base
    return SRVGGNetCompact()


def run_internal_srvgg_anime_upscale(input_path: Path, output_path: Path, settings: dict):
    import torch
    clear_vram()
    native_scale = 4
    if int(settings['scale']) not in [4, 8, 2]:
        raise RuntimeError('Internal Real-ESRGAN Anime fallback supports requested output scales 2x, 4x, or 8x via native 4x plus resize.')
    precision = effective_precision(settings['precision'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tile = effective_tile(settings['tile'], True)
    weight_path = download_url(MODEL_REGISTRY['Real-ESRGAN Anime']['weights']['x4'], CACHE_DIR/'weights/RealESRGAN_x4plus_anime_6B.pth')
    model = build_internal_srvggnet(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4, act_type='prelu')
    model = load_internal_state_dict(model, weight_path, device)
    if precision == 'FP16' and torch.cuda.is_available():
        model = model.half()
    img, alpha = load_cv2_image_exif_safe(input_path, keep_alpha=True)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(np.transpose(rgb, (2, 0, 1))).float().unsqueeze(0) / 255.0
    tensor = tensor.to(device)
    if precision == 'FP16' and torch.cuda.is_available():
        tensor = tensor.half()
    try:
        out = internal_model_forward_tiled(model, tensor, native_scale, tile=tile, tile_pad=10)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower() and (not tile or tile > 256):
            print('⚠️ CUDA OOM in internal Anime SRVGG engine; retrying with tile=256...')
            clear_vram(aggressive=True)
            out = internal_model_forward_tiled(model, tensor, native_scale, tile=256, tile_pad=10)
            tile = 256
        elif precision == 'FP16':
            print('⚠️ FP16 internal Anime SRVGG failed; retrying FP32...')
            retry_settings = copy.deepcopy(settings)
            retry_settings['precision'] = 'FP32'
            return run_internal_srvgg_anime_upscale(input_path, output_path, retry_settings)
        else:
            raise
    out = out.float().clamp_(0, 1).detach().cpu().numpy()[0]
    out_rgb = np.transpose(out, (1, 2, 0))
    out_bgr = cv2.cvtColor((out_rgb * 255.0).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
    requested_scale = int(settings['scale'])
    if requested_scale != native_scale:
        original_w, original_h = Image.open(input_path).size
        out_bgr = cv2.resize(out_bgr, (original_w * requested_scale, original_h * requested_scale), interpolation=cv2.INTER_LANCZOS4)
    out_bgr = merge_resized_alpha_if_needed(out_bgr, alpha)
    save_cv2_image(output_path, out_bgr)
    del model, tensor
    clear_vram()
    return {'effective_precision': precision, 'tile': tile, 'native_scale': native_scale, 'engine': 'internal_srvgg_anime_no_basicsr'}

def run_realesrgan_like(input_path: Path, output_path: Path, settings: dict, ultrasharp=False):
    clear_vram()
    model_name = settings['model']
    # Current Colab Python 3.13 breaks BasicSR metadata generation. For RRDB models,
    # skip the package stack entirely and use the internal PyTorch RRDB engine.
    if sys.version_info >= (3, 13):
        if model_name == 'Real-ESRGAN' or ultrasharp:
            print('Python 3.13 runtime detected; using internal RRDB engine instead of BasicSR/RealESRGAN package install.')
            return run_internal_rrdb_upscale(input_path, output_path, settings, ultrasharp=ultrasharp)
        if model_name == 'Real-ESRGAN Anime':
            print('Python 3.13 runtime detected; using internal Anime SRVGG engine instead of BasicSR/RealESRGAN package install.')
            return run_internal_srvgg_anime_upscale(input_path, output_path, settings)
    try:
        ensure_realesrgan_stack()
        patch_torchvision_functional_tensor()
        import torch
        from realesrgan import RealESRGANer
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan.archs.srvgg_arch import SRVGGNetCompact
        from huggingface_hub import hf_hub_download
    except Exception as e:
        if model_name == 'Real-ESRGAN' or ultrasharp:
            print(f'⚠️ Real-ESRGAN package stack unavailable; using internal RRDB fallback engine. Reason: {e}')
            return run_internal_rrdb_upscale(input_path, output_path, settings, ultrasharp=ultrasharp)
        raise

    scale_req = int(settings['scale'])
    precision = effective_precision(settings['precision'])
    half = (precision == 'FP16' and torch.cuda.is_available())
    tile = effective_tile(settings['tile'], True)

    if ultrasharp:
        native_scale = 4
        weight_path = hf_hub_download(repo_id='Kim2091/UltraSharp', filename='4x-UltraSharp.pth', cache_dir=str(CACHE_DIR/'hf'))
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    elif model_name == 'Real-ESRGAN Anime':
        native_scale = 4
        weight_path = download_url(MODEL_REGISTRY[model_name]['weights']['x4'], CACHE_DIR/'weights/RealESRGAN_x4plus_anime_6B.pth')
        model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4, act_type='prelu')
    else:
        # RealESRGAN_x2plus is a true x2 RRDBNet. Keep both the model arch scale and RealESRGANer scale at 2.
        if scale_req == 2:
            native_scale = 2
            key = 'x2'
            weight_path = download_url(MODEL_REGISTRY[model_name]['weights'][key], CACHE_DIR/'weights/RealESRGAN_x2plus.pth')
            model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
        else:
            native_scale = 4
            key = 'x4'
            weight_path = download_url(MODEL_REGISTRY[model_name]['weights'][key], CACHE_DIR/'weights/RealESRGAN_x4plus.pth')
            model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)

    upsampler = RealESRGANer(
        scale=native_scale,
        model_path=str(weight_path),
        dni_weight=None,
        model=model,
        tile=tile,
        tile_pad=10,
        pre_pad=0,
        half=half,
        gpu_id=0 if torch.cuda.is_available() else None,
    )

    # RealESRGANer follows the official OpenCV path: input/output arrays are BGR/BGRA.
    # Load via PIL first to apply EXIF orientation, then convert to BGR/BGRA for RealESRGANer.
    img, alpha = load_cv2_image_exif_safe(input_path, keep_alpha=True)
    try:
        output, _ = upsampler.enhance(img, outscale=float(scale_req))
    except RuntimeError as e:
        if 'CUDA out of memory' in str(e) and tile == 0:
            print('⚠️ CUDA OOM; retrying with tile=256...')
            clear_vram()
            upsampler.tile = 256
            output, _ = upsampler.enhance(img, outscale=float(scale_req))
        elif half:
            print('⚠️ FP16 failed; retrying FP32...')
            clear_vram()
            retry_settings = copy.deepcopy(settings)
            retry_settings['precision'] = 'FP32'
            return run_realesrgan_like(input_path, output_path, retry_settings, ultrasharp=ultrasharp)
        else:
            raise
    output = merge_resized_alpha_if_needed(output, alpha)
    save_cv2_image(output_path, output)

    del upsampler, model
    clear_vram()
    return {'effective_precision': precision, 'tile': tile, 'native_scale': native_scale}



def ensure_swinir_stack():
    # SwinIR needs timm and the official network_swinir.py only; no BasicSR package.
    if importlib.util.find_spec('timm') is None:
        pip_install('timm')


def load_official_swinir_class():
    ensure_swinir_stack()
    arch_dir = CACHE_DIR / 'swinir_official'
    arch_dir.mkdir(parents=True, exist_ok=True)
    arch_path = arch_dir / 'network_swinir.py'
    if not arch_path.exists() or arch_path.stat().st_size < 10000:
        url = 'https://raw.githubusercontent.com/JingyunLiang/SwinIR/main/models/network_swinir.py'
        print(f'Downloading official SwinIR architecture -> {arch_path}')
        download_url(url, arch_path, min_bytes=10000, retries=3)
    import importlib.util as _importlib_util
    spec = _importlib_util.spec_from_file_location('swinir_network_official', str(arch_path))
    module = _importlib_util.module_from_spec(spec)
    sys.modules['swinir_network_official'] = module
    spec.loader.exec_module(module)
    return module.SwinIR


def build_swinir_real_sr_x4_model():
    SwinIR = load_official_swinir_class()
    return SwinIR(
        upscale=4, in_chans=3, img_size=64, window_size=8, img_range=1.0,
        depths=[6, 6, 6, 6, 6, 6], embed_dim=180,
        num_heads=[6, 6, 6, 6, 6, 6], mlp_ratio=2,
        upsampler='nearest+conv', resi_connection='1conv'
    )


def swinir_forward_tiled(model, tensor, scale=4, tile=0, tile_overlap=32, window_size=8):
    import torch
    import torch.nn.functional as F
    b, c, h_old, w_old = tensor.shape
    h_pad = (h_old // window_size + 1) * window_size - h_old if h_old % window_size else 0
    w_pad = (w_old // window_size + 1) * window_size - w_old if w_old % window_size else 0
    if h_pad or w_pad:
        tensor = F.pad(tensor, (0, w_pad, 0, h_pad), mode='reflect')
    _, _, h, w = tensor.shape
    if not tile or tile <= 0:
        with torch.no_grad():
            out = model(tensor)
        return out[:, :, :h_old * scale, :w_old * scale]
    tile = min(tile, h, w)
    stride = max(tile - tile_overlap, 1)
    E = torch.zeros((b, c, h * scale, w * scale), dtype=tensor.dtype, device=tensor.device)
    W = torch.zeros_like(E)
    for y in range(0, h, stride):
        for x in range(0, w, stride):
            y0 = min(y, h - tile)
            x0 = min(x, w - tile)
            in_patch = tensor[:, :, y0:y0 + tile, x0:x0 + tile]
            with torch.no_grad():
                out_patch = model(in_patch)
            out_y0, out_x0 = y0 * scale, x0 * scale
            E[:, :, out_y0:out_y0 + tile * scale, out_x0:out_x0 + tile * scale] += out_patch
            W[:, :, out_y0:out_y0 + tile * scale, out_x0:out_x0 + tile * scale] += 1
    out = E.div_(W)
    return out[:, :, :h_old * scale, :w_old * scale]


def run_swinir(input_path: Path, output_path: Path, settings: dict):
    import torch
    clear_vram()
    if int(settings['scale']) != 4:
        raise RuntimeError('SwinIR official real-world model in this notebook supports native 4x only.')
    precision = effective_precision(settings['precision'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tile = effective_tile(settings['tile'], True)
    if tile == 0 and HARDWARE.get('vram_total_gb', 0) and HARDWARE.get('vram_total_gb', 0) < 24:
        tile = 256
        print('SwinIR Auto tile selected 256 for Colab VRAM safety.')
    model = build_swinir_real_sr_x4_model().to(device).eval()
    weight_path = download_url(MODEL_REGISTRY['SwinIR']['weights']['x4'], CACHE_DIR/'weights/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth', min_bytes=50*1024*1024)
    loadnet = torch.load(str(weight_path), map_location=device)
    state = loadnet.get('params_ema', loadnet.get('params', loadnet)) if isinstance(loadnet, dict) else loadnet
    model.load_state_dict(state, strict=True)
    if precision == 'FP16' and torch.cuda.is_available():
        model = model.half()
    img, alpha = load_cv2_image_exif_safe(input_path, keep_alpha=True)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(np.transpose(rgb, (2, 0, 1))).float().unsqueeze(0) / 255.0
    tensor = tensor.to(device)
    if precision == 'FP16' and torch.cuda.is_available():
        tensor = tensor.half()
    try:
        out = swinir_forward_tiled(model, tensor, scale=4, tile=tile, tile_overlap=32, window_size=8)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower() and (not tile or tile > 128):
            print('⚠️ CUDA OOM in SwinIR; retrying with tile=128...')
            clear_vram(aggressive=True)
            out = swinir_forward_tiled(model, tensor, scale=4, tile=128, tile_overlap=16, window_size=8)
            tile = 128
        elif precision == 'FP16':
            print('⚠️ FP16 SwinIR failed; retrying FP32...')
            retry_settings = copy.deepcopy(settings)
            retry_settings['precision'] = 'FP32'
            return run_swinir(input_path, output_path, retry_settings)
        else:
            raise
    out = out.float().clamp_(0, 1).detach().cpu().numpy()[0]
    out_rgb = np.transpose(out, (1, 2, 0))
    out_bgr = cv2.cvtColor((out_rgb * 255.0).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
    out_bgr = merge_resized_alpha_if_needed(out_bgr, alpha)
    save_cv2_image(output_path, out_bgr)
    del model, tensor
    clear_vram()
    return {'effective_precision': precision, 'tile': tile, 'native_scale': 4, 'engine': 'official_swinir_arch_lazy'}



def ensure_realcugan_binary():
    tool_root = CACHE_DIR / 'realcugan-ncnn-vulkan-20220728-ubuntu'
    exe = tool_root / 'realcugan-ncnn-vulkan'
    if exe.exists():
        exe.chmod(0o755)
        return tool_root, exe
    zip_path = CACHE_DIR / 'weights' / 'realcugan-ncnn-vulkan-20220728-ubuntu.zip'
    download_url(
        MODEL_REGISTRY['Real-CUGAN']['weights']['ubuntu_zip'],
        zip_path,
        min_bytes=10*1024*1024,
        expected_bytes=EXPECTED_BINARY_BYTES['realcugan-ncnn-vulkan-20220728-ubuntu.zip']
    )
    print(f'Extracting Real-CUGAN binary package -> {CACHE_DIR}')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(CACHE_DIR)
    if not exe.exists():
        # Some zips contain a top-level directory with a slightly different name; search defensively.
        matches = list(CACHE_DIR.glob('**/realcugan-ncnn-vulkan'))
        if not matches:
            raise RuntimeError('Real-CUGAN binary was not found after extracting the release zip.')
        exe = matches[0]
        tool_root = exe.parent
    exe.chmod(0o755)
    return tool_root, exe


def run_realcugan(input_path: Path, output_path: Path, settings: dict):
    clear_vram()
    scale = int(settings['scale'])
    if scale not in [2, 4]:
        raise RuntimeError('Real-CUGAN adapter currently supports 2x and 4x from the Colab form.')
    tool_root, exe = ensure_realcugan_binary()
    tile = effective_tile(settings['tile'], True)
    tile_arg = str(tile if tile else 0)
    preset = settings.get('quality_preset', 'Balanced')
    model_dir = tool_root / ('models-se' if preset == 'Fast' else 'models-pro')
    if not model_dir.exists():
        # Fallback to any bundled model folder.
        candidates = [p for p in tool_root.iterdir() if p.is_dir() and p.name.startswith('models')]
        if not candidates:
            raise RuntimeError('No bundled Real-CUGAN model directory was found in the binary package.')
        model_dir = candidates[0]
    cmd = [
        str(exe), '-i', str(input_path), '-o', str(output_path),
        '-s', str(scale), '-t', tile_arg, '-n', '-1', '-m', str(model_dir), '-f', 'png'
    ]
    # Prefer CPU mode on Colab because Vulkan GPU availability is inconsistent in notebooks.
    # If the binary does not support CPU on the current image, the error is reported clearly.
    cmd += ['-g', '-1']
    print('Running Real-CUGAN command:', ' '.join(cmd))
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=1800)
    if p.stdout:
        print(p.stdout[-4000:])
    if p.returncode != 0 or not Path(output_path).exists():
        raise RuntimeError(f'Real-CUGAN binary failed with code {p.returncode}. Output:\n{p.stdout[-4000:]}')
    clear_vram()
    return {'effective_precision': 'External binary', 'tile': tile_arg, 'native_scale': scale, 'engine': 'realcugan_ncnn_vulkan_cpu'}



def ensure_spandrel_stack(extra_arches=False):
    if importlib.util.find_spec('spandrel') is None:
        # Spandrel currently advertises Python <3.12 on PyPI, but the lightweight
        # architecture loaders used here are pure PyTorch and can be attempted on
        # current Colab. Keep this lazy and isolated so other models still work if
        # pip or runtime compatibility fails.
        pip_install('spandrel==0.4.2', extra_args='--ignore-requires-python')
    if extra_arches:
        if importlib.util.find_spec('spandrel_extra_arches') is None:
            pip_install('spandrel-extra-arches', extra_args='--ignore-requires-python')
        import spandrel_extra_arches
        spandrel_extra_arches.install()


def spandrel_forward_tiled(model, tensor, scale=4, tile=0, tile_overlap=32):
    import torch
    b, c, h, w = tensor.shape
    if not tile or tile <= 0:
        with torch.no_grad():
            return model(tensor)
    tile = min(tile, h, w)
    stride = max(tile - tile_overlap, 1)
    E = torch.zeros((b, c, h * scale, w * scale), dtype=tensor.dtype, device=tensor.device)
    W = torch.zeros_like(E)
    ys = list(range(0, h, stride))
    xs = list(range(0, w, stride))
    for y in ys:
        for x in xs:
            y0 = min(y, h - tile)
            x0 = min(x, w - tile)
            in_patch = tensor[:, :, y0:y0 + tile, x0:x0 + tile]
            with torch.no_grad():
                out_patch = model(in_patch)
            out_y0, out_x0 = y0 * scale, x0 * scale
            E[:, :, out_y0:out_y0 + out_patch.shape[-2], out_x0:out_x0 + out_patch.shape[-1]] += out_patch
            W[:, :, out_y0:out_y0 + out_patch.shape[-2], out_x0:out_x0 + out_patch.shape[-1]] += 1
    return E.div_(W)


def run_spandrel_hf(input_path: Path, output_path: Path, settings: dict):
    import torch
    from huggingface_hub import hf_hub_download
    ensure_spandrel_stack()
    from spandrel import ImageModelDescriptor, ModelLoader
    clear_vram()
    model_name = settings['model']
    meta = MODEL_REGISTRY[model_name]
    scale = int(settings['scale'])
    if scale not in meta.get('native_scales', [4]):
        raise RuntimeError(f'{model_name} Spandrel adapter supports native scale(s) {meta.get("native_scales")}.')
    precision = effective_precision(settings['precision'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tile = effective_tile(settings['tile'], True)
    if tile == 0 and HARDWARE.get('vram_total_gb', 0) and HARDWARE.get('vram_total_gb', 0) < 24:
        tile = 256
        print(f'{model_name} Auto tile selected 256 for Colab VRAM safety.')
    weight_path = hf_hub_download(repo_id=meta['weights']['hf_repo'], filename=meta['weights']['hf_file'], cache_dir=str(CACHE_DIR/'hf'))
    descriptor = ModelLoader().load_from_file(weight_path)
    if not isinstance(descriptor, ImageModelDescriptor):
        raise RuntimeError(f'{model_name} checkpoint did not load as an image-to-image Spandrel model.')
    descriptor = descriptor.to(device).eval()
    if precision == 'FP16' and torch.cuda.is_available():
        descriptor = descriptor.half()
    # Prefer descriptor scale if exposed, but keep registry as source of output tiling scale.
    actual_scale = int(getattr(descriptor, 'scale', scale) or scale)
    img, alpha = load_cv2_image_exif_safe(input_path, keep_alpha=True)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(np.transpose(rgb, (2, 0, 1))).float().unsqueeze(0) / 255.0
    tensor = tensor.to(device)
    if precision == 'FP16' and torch.cuda.is_available():
        tensor = tensor.half()
    try:
        out = spandrel_forward_tiled(descriptor, tensor, scale=actual_scale, tile=tile, tile_overlap=32)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower() and (not tile or tile > 128):
            print(f'⚠️ CUDA OOM in {model_name}; retrying with tile=128...')
            clear_vram(aggressive=True)
            out = spandrel_forward_tiled(descriptor, tensor, scale=actual_scale, tile=128, tile_overlap=16)
            tile = 128
        elif precision == 'FP16':
            print(f'⚠️ FP16 {model_name} failed; retrying FP32...')
            retry_settings = copy.deepcopy(settings)
            retry_settings['precision'] = 'FP32'
            return run_spandrel_hf(input_path, output_path, retry_settings)
        else:
            raise
    out = out.float().clamp_(0, 1).detach().cpu().numpy()[0]
    out_rgb = np.transpose(out, (1, 2, 0))
    out_bgr = cv2.cvtColor((out_rgb * 255.0).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
    out_bgr = merge_resized_alpha_if_needed(out_bgr, alpha)
    save_cv2_image(output_path, out_bgr)
    del descriptor, tensor
    clear_vram()
    return {'effective_precision': precision, 'tile': tile, 'native_scale': actual_scale, 'engine': 'spandrel_hf_auto_loader'}


def run_aurasr(input_path: Path, output_path: Path, settings: dict):
    clear_vram()
    ensure_aura_sr()
    import torch
    from aura_sr import AuraSR
    if int(settings['scale']) != 4:
        raise RuntimeError('AuraSR supports 4x only in its official API.')
    precision = effective_precision(settings['precision'])
    img = Image.open(input_path).convert('RGB')
    try:
        model = AuraSR.from_pretrained('fal-ai/AuraSR')
    except TypeError:
        model = AuraSR.from_pretrained()
    if settings.get('generative_detail'):
        if hasattr(model, 'upscale_4x_overlapped'):
            out = model.upscale_4x_overlapped(img)
        else:
            out = model.upscale_4x(img)
    else:
        out = model.upscale_4x(img)
    out.save(output_path)
    del model
    clear_vram()
    return {'effective_precision': precision, 'tile': 'Not supported', 'native_scale': 4}


def apply_preserve_original(input_path: Path, output_path: Path, preserve_percent: int):
    """Blend the SR result with a high-quality resized original.

    preserve_percent = 0 keeps the model output unchanged.
    preserve_percent = 100 returns a pure Lanczos-resized original.
    This is a conservative fidelity control, especially useful for generative SR.
    """
    try:
        preserve_percent = int(preserve_percent or 0)
    except Exception:
        preserve_percent = 0
    preserve_percent = max(0, min(100, preserve_percent))
    if preserve_percent <= 0:
        return 'OFF'

    sr = Image.open(output_path)
    mode = 'RGBA' if sr.mode == 'RGBA' else 'RGB'
    sr_img = sr.convert(mode)
    original = ImageOps.exif_transpose(Image.open(input_path)).convert(mode)
    original_up = original.resize(sr_img.size, Image.Resampling.LANCZOS)
    alpha = preserve_percent / 100.0
    blended = Image.blend(sr_img, original_up, alpha)
    blended.save(output_path)
    return f'{preserve_percent}% blend with resized original'


def ensure_face_restoration_stack(method='GFPGAN'):
    # Face restoration post-process is kept separate from the Real-ESRGAN/BasicSR stack.
    # On current Colab Python 3.13, BasicSR setup is fragile; Spandrel loads
    # the face network weights while facexlib handles face detection/alignment.
    ensure_spandrel_stack(extra_arches=(method == 'CodeFormer'))
    if importlib.util.find_spec('facexlib') is None:
        pip_install('facexlib', extra_args='--no-build-isolation')


def _tensor_to_bgr_uint8(tensor, min_max=(-1, 1)):
    import torch
    tensor = tensor.detach().float().cpu().clamp_(*min_max)
    tensor = (tensor - min_max[0]) / (min_max[1] - min_max[0])
    arr = tensor.squeeze(0).clamp_(0, 1).numpy()
    arr = np.transpose(arr, (1, 2, 0))
    arr = (arr * 255.0).round().astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def apply_gfpgan_if_requested(image_path: Path, settings: dict):
    method = settings.get('face_restoration')
    if method not in ('GFPGAN', 'CodeFormer'):
        return None
    print(f'Applying {method} face restoration as post-process...')
    ensure_face_restoration_stack(method)
    import torch
    from torchvision.transforms.functional import normalize
    from spandrel import ModelLoader
    from facexlib.utils.face_restoration_helper import FaceRestoreHelper

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if method == 'GFPGAN':
        weight_url = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth'
        weight_file = 'GFPGANv1.3.pth'
    else:
        weight_url = 'https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth'
        weight_file = 'codeformer.pth'
    weight_path = download_url(weight_url, CACHE_DIR/'weights'/weight_file)
    descriptor = ModelLoader().load_from_file(str(weight_path))
    net = getattr(descriptor, 'model', descriptor).to(device).eval()
    img, alpha = load_cv2_image_exif_safe(image_path, keep_alpha=True)
    face_helper = FaceRestoreHelper(
        upscale=1,
        face_size=512,
        crop_ratio=(1, 1),
        det_model='retinaface_resnet50',
        save_ext='png',
        use_parse=True,
        device=device,
        model_rootpath=str(CACHE_DIR/'facexlib')
    )
    face_helper.clean_all()
    face_helper.read_image(img)
    face_helper.get_face_landmarks_5(only_center_face=False, eye_dist_threshold=5)
    face_helper.align_warp_face()
    if not face_helper.cropped_faces:
        print(f'{method}: no suitable faces found; image left unchanged.')
        del net, face_helper
        clear_vram()
        return f'{method} skipped: no suitable faces found'
    print(f'{method}: restoring {len(face_helper.cropped_faces)} detected face(s), paste_back=True.')
    for cropped_face in face_helper.cropped_faces:
        face_rgb = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        cropped_face_t = torch.from_numpy(np.transpose(face_rgb, (2, 0, 1))).float()
        normalize(cropped_face_t, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5), inplace=True)
        cropped_face_t = cropped_face_t.unsqueeze(0).to(device)
        try:
            with torch.no_grad():
                if method == 'GFPGAN':
                    output = net(cropped_face_t, return_rgb=False, weight=0.5)[0]
                else:
                    # CodeFormer w: lower is more restorative, higher preserves identity/texture.
                    output = net(cropped_face_t, w=0.7, adain=True)[0]
            restored_face = _tensor_to_bgr_uint8(output, min_max=(-1, 1))
        except RuntimeError as e:
            print(f'⚠️ {method} inference failed for one face; keeping original crop. Reason: {e}')
            restored_face = cropped_face
        face_helper.add_restored_face(restored_face.astype(np.uint8))
    face_helper.get_inverse_affine(None)
    restored = face_helper.paste_faces_to_input_image(upsample_img=img)
    restored = merge_resized_alpha_if_needed(restored, alpha)
    save_cv2_image(image_path, restored)
    del net, face_helper
    clear_vram()
    return f'{method} applied via Spandrel + facexlib'


def upscale_one_model(input_path: Path, input_info: dict, model_name: str, settings: dict, suffix='', defer_face_restoration=False):
    meta = MODEL_REGISTRY.get(model_name, {})
    model_settings = copy.deepcopy(settings)
    model_settings['model'] = model_name
    local_settings, warns = normalize_settings(model_settings)
    if meta.get('status') == 'unavailable':
        raise RuntimeError(f"{model_name}: Unavailable on current runtime — {meta.get('reason')}")

    out_name = f"upscaled{suffix}.png" if suffix else 'upscaled.png'
    output_path = OUTPUT_DIR / out_name
    print(f"\n===== UPSCALE: {model_name} =====")
    print('Settings:', json.dumps(local_settings, indent=2))
    t0 = time.time()
    if meta.get('implementation') == 'aura_sr':
        proc_info = run_aurasr(input_path, output_path, local_settings)
    elif meta.get('implementation') == 'esrgan_rrdb_hf':
        proc_info = run_realesrgan_like(input_path, output_path, local_settings, ultrasharp=True)
    elif meta.get('implementation') == 'python_api':
        proc_info = run_realesrgan_like(input_path, output_path, local_settings, ultrasharp=False)
    elif meta.get('implementation') == 'swinir_official':
        proc_info = run_swinir(input_path, output_path, local_settings)
    elif meta.get('implementation') == 'realcugan_ncnn':
        proc_info = run_realcugan(input_path, output_path, local_settings)
    elif meta.get('implementation') == 'spandrel_hf':
        proc_info = run_spandrel_hf(input_path, output_path, local_settings)
    else:
        raise RuntimeError(f'{model_name} has no runnable implementation in this notebook.')

    preserve_note = apply_preserve_original(input_path, output_path, local_settings.get('preserve_original_percent', 0))
    if defer_face_restoration and local_settings.get('face_restoration') in ('GFPGAN', 'CodeFormer'):
        face_note = 'Deferred until after compare upscales'
    else:
        face_note = apply_gfpgan_if_requested(output_path, local_settings)
    elapsed = round(time.time() - t0, 2)
    out_img = Image.open(output_path)
    result = {
        'model': model_name,
        'output_path': str(output_path),
        'settings': local_settings,
        'setting_warnings': warns,
        'processing_info': {**proc_info, 'elapsed_sec': elapsed, 'preserve_original': preserve_note, 'face_restoration': face_note},
        'input_info': input_info,
        'output_resolution': list(out_img.size),
        'errors': [],
    }
    print(f"✅ Done: {output_path} ({out_img.size[0]}x{out_img.size[1]}) in {elapsed}s")
    clear_vram()
    return result

UPSCALE_RESULTS = []
INPUT_IMAGE_PATH = None
INPUT_INFO = None

if RUN_UPSCALE:
    if (not MODEL_SETTINGS.get('compare_mode')) and NORMALIZED_SETTINGS is None:
        raise RuntimeError('Invalid model/settings. Fix CELL 4 selection first.')
    INPUT_IMAGE_PATH, INPUT_INFO = upload_one_image()
    base_settings = copy.deepcopy(MODEL_SETTINGS)  # raw UI settings; each model is normalized independently below.
    selected_models = base_settings['compare_models'] if base_settings.get('compare_mode') else [base_settings['model']]
    original_aggressive_cleanup = bool(globals().get('AGGRESSIVE_CUDA_CLEANUP', True))
    if base_settings.get('compare_mode'):
        # In Compare Mode, avoid expensive ipc_collect between every model; do aggressive cleanup once at the end.
        globals()['AGGRESSIVE_CUDA_CLEANUP'] = False
    if len(selected_models) > 4:
        print(f'⚠️ Compare Mode has {len(selected_models)} models. Running only the first 4 to reduce Colab time/VRAM risk.')
        selected_models = selected_models[:4]
    # Filter and run sequentially; never load all models at once.
    used_slugs = {}
    for m in selected_models:
        try:
            suffix = '_' + safe_model_slug(m, used_slugs) if (base_settings.get('compare_mode') or len(selected_models) > 1) else ''
            res = upscale_one_model(INPUT_IMAGE_PATH, INPUT_INFO, m, base_settings, suffix=suffix, defer_face_restoration=base_settings.get('compare_mode'))
            UPSCALE_RESULTS.append(res)
        except Exception as e:
            err_settings = copy.deepcopy(base_settings)
            err_settings['model'] = m
            err = {'model': m, 'errors': [str(e)], 'output_path': None, 'settings': err_settings, 'input_info': INPUT_INFO}
            UPSCALE_RESULTS.append(err)
            print(f"❌ {m}: {e}")
        finally:
            unload_between_models(m, aggressive=False)
    if base_settings.get('compare_mode') and base_settings.get('face_restoration') in ('GFPGAN', 'CodeFormer'):
        print(f"Applying {base_settings.get('face_restoration')} after all compare upscales...")
        for res in UPSCALE_RESULTS:
            if res.get('output_path'):
                try:
                    note = apply_gfpgan_if_requested(Path(res['output_path']), res['settings'])
                    res['processing_info']['face_restoration'] = note
                    res['output_resolution'] = list(Image.open(res['output_path']).size)
                except Exception as e:
                    res.setdefault('errors', []).append(f"{base_settings.get('face_restoration')} post-compare failed: {e}")
                    print(f"❌ {base_settings.get('face_restoration')} post-compare failed for {res.get('model')}: {e}")
                finally:
                    unload_between_models(f"GFPGAN {res.get('model')}", aggressive=False)
    if base_settings.get('compare_mode'):
        globals()['AGGRESSIVE_CUDA_CLEANUP'] = original_aggressive_cleanup
        print('Final aggressive CUDA cleanup after Compare Mode.')
        clear_vram(aggressive=True)
else:
    print('RUN_UPSCALE is False. Set to True and run this cell when ready.')


In [ ]:

# CELL 6 — Quality Control Engine
#@title CELL 6 — Quality Control Engine (MUSIQ, NIQE, BRISQUE, Laplacian, 2D FFT)
import tempfile

IQA_CACHE = {}

def resize_for_iqa(img: Image.Image, max_side=1280):
    img = ImageOps.exif_transpose(img).convert('RGB')
    w, h = img.size
    if max(w, h) > max_side:
        ratio = max_side / max(w, h)
        img = img.resize((int(w*ratio), int(h*ratio)), Image.Resampling.LANCZOS)
    return img


def pyiqa_score(metric_name: str, img: Image.Image):
    tmp_name = None
    try:
        if globals().get('PYIQA_AVAILABLE') is False:
            return None, 'pyiqa was not installed successfully in CELL 2'
        import torch, pyiqa
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if metric_name not in IQA_CACHE:
            IQA_CACHE[metric_name] = pyiqa.create_metric(metric_name, device=device)
        metric = IQA_CACHE[metric_name]
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
            tmp_name = tmp.name
            resize_for_iqa(img).save(tmp.name)
        val = metric(tmp_name)
        if hasattr(val, 'detach'):
            val = float(val.detach().cpu().numpy().squeeze())
        else:
            val = float(val)
        return round(val, 4), None
    except Exception as e:
        return None, str(e)
    finally:
        if tmp_name:
            try:
                os.unlink(tmp_name)
            except Exception:
                pass


def laplacian_variance(img: Image.Image):
    arr = np.array(img.convert('RGB'))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    return round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 4)


def fft_analysis(img: Image.Image):
    gray = np.array(resize_for_iqa(img, max_side=1024).convert('L')).astype(np.float32) / 255.0
    h, w = gray.shape
    window = np.outer(np.hanning(h), np.hanning(w)).astype(np.float32)
    f = np.fft.fftshift(np.fft.fft2(gray * window))
    mag = np.log1p(np.abs(f))
    yy, xx = np.indices((h, w))
    cy, cx = h//2, w//2
    r = np.sqrt(((yy-cy)/(h/2))**2 + ((xx-cx)/(w/2))**2)
    total = float(mag.sum() + 1e-8)
    hf_ratio = float(mag[r > 0.55].sum() / total)
    mf_ratio = float(mag[(r > 0.25) & (r <= 0.55)].sum() / total)
    # line energy: ringing/grid/repeated texture often appears as row/column spikes.
    center_band = 3
    row_energy = float(mag[cy-center_band:cy+center_band+1, :].sum() / total)
    col_energy = float(mag[:, cx-center_band:cx+center_band+1].sum() / total)
    line_energy = row_energy + col_energy
    # peak anomaly outside DC region
    outer = mag[r > 0.18]
    peak_z = float((outer.max() - outer.mean()) / (outer.std() + 1e-8)) if outer.size else 0.0
    status = 'PASS'
    reasons = []
    if hf_ratio > 0.18:
        status = 'WARNING'; reasons.append('High-frequency energy is elevated')
    if line_energy > 0.18 or peak_z > 18:
        status = 'WARNING'; reasons.append('Possible periodic/grid or ringing pattern')
    if hf_ratio > 0.25 or line_energy > 0.25 or peak_z > 25:
        status = 'FAIL'; reasons.append('Strong synthetic high-frequency artifact pattern')
    return {
        'status': status,
        'high_frequency_ratio': round(hf_ratio, 5),
        'mid_frequency_ratio': round(mf_ratio, 5),
        'line_energy_ratio': round(line_energy, 5),
        'peak_z': round(peak_z, 3),
        'reasons': reasons or ['No significant FFT artifact detected'],
    }


def compute_quality_metrics(image_path: str):
    img = Image.open(image_path)
    metrics = {}
    errors = {}
    for m in ['musiq', 'niqe', 'brisque']:
        score, err = pyiqa_score(m, img)
        metrics[musiq_name(m)] = score
        if err:
            errors[musiq_name(m)] = err
    metrics['laplacian_variance'] = laplacian_variance(img)
    metrics['fft'] = fft_analysis(img)
    if errors:
        metrics['metric_errors'] = errors
    return metrics


def musiq_name(m):
    return {'musiq':'MUSIQ','niqe':'NIQE','brisque':'BRISQUE'}.get(m, m)


def classify_metric(name, value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return 'WARNING', 'Metric unavailable'
    if name == 'MUSIQ':
        if value >= 65: return 'PASS', 'MUSIQ technical/aesthetic score in pass range'
        if value >= 50: return 'WARNING', 'MUSIQ in warning range'
        return 'FAIL', 'MUSIQ below baseline fail range'
    if name == 'NIQE':
        if 3.0 <= value <= 5.5: return 'PASS', 'NIQE in baseline pass range'
        if 5.5 < value <= 7.0 or value < 3.0: return 'WARNING', 'NIQE outside ideal band; interpret carefully for AI art'
        return 'FAIL', 'NIQE high; likely natural-scene quality degradation'
    if name == 'BRISQUE':
        if value < 30: return 'PASS', 'BRISQUE in pass range'
        if value <= 45: return 'WARNING', 'BRISQUE in warning range'
        return 'FAIL', 'BRISQUE in fail range'
    if name == 'laplacian_variance':
        if 300 <= value <= 1500: return 'PASS', 'Sharpness within target range'
        if 150 <= value < 300 or 1500 < value <= 3000: return 'WARNING', 'Sharpness outside target; possible blur or oversharpening'
        return 'FAIL', 'Sharpness extreme; likely blur or oversharpening/artifacts'
    return 'PASS', ''


def delta_metrics(before, after):
    out = {}
    for k in ['MUSIQ','NIQE','BRISQUE','laplacian_variance']:
        b, a = before.get(k), after.get(k)
        out[k] = None if b is None or a is None else round(a - b, 4)
    try:
        out['fft_high_frequency_ratio'] = round(after['fft']['high_frequency_ratio'] - before['fft']['high_frequency_ratio'], 5)
        out['fft_line_energy_ratio'] = round(after['fft']['line_energy_ratio'] - before['fft']['line_energy_ratio'], 5)
    except Exception:
        pass
    return out


def interpret_quality(before, after, settings):
    reasons, warnings_list, errors = [], [], []
    statuses = []
    delta = delta_metrics(before, after)

    # Raw threshold statuses after upscale
    for k in ['MUSIQ','NIQE','BRISQUE','laplacian_variance']:
        st, reason = classify_metric(k, after.get(k))
        statuses.append(st)
        (errors if st == 'FAIL' else warnings_list if st == 'WARNING' else reasons).append(reason)

    # Direction-aware before/after interpretation — no averaging.
    if delta.get('MUSIQ') is not None:
        if delta['MUSIQ'] > 1: reasons.append('MUSIQ improved')
        elif delta['MUSIQ'] < -3: errors.append('MUSIQ degraded after upscale')
        else: warnings_list.append('MUSIQ changed only slightly')
    if delta.get('NIQE') is not None:
        if delta['NIQE'] < -0.2: reasons.append('NIQE improved')
        elif delta['NIQE'] > 0.5: errors.append('NIQE degraded after upscale')
    if delta.get('BRISQUE') is not None:
        if delta['BRISQUE'] < -1: reasons.append('BRISQUE improved')
        elif delta['BRISQUE'] > 3: errors.append('BRISQUE degraded after upscale')

    lap_b, lap_a = before.get('laplacian_variance'), after.get('laplacian_variance')
    if lap_b and lap_a:
        ratio = lap_a / max(lap_b, 1e-6)
        if lap_a > 3000 or (ratio > 4 and lap_a > 1500):
            errors.append('Sharpness increased excessively; possible oversharpening / edge halos')
        elif ratio > 2.5 and lap_a > 1500:
            warnings_list.append('Sharpness increased strongly; inspect for ringing/halo artifacts')
        elif ratio < 0.65:
            warnings_list.append('Sharpness decreased; possible plastic/smoothed texture')
        else:
            reasons.append('Sharpness change is within expected range')

    fft_after = after.get('fft', {})
    if fft_after.get('status') == 'FAIL':
        errors.extend(fft_after.get('reasons', []))
    elif fft_after.get('status') == 'WARNING':
        warnings_list.extend(fft_after.get('reasons', []))
    else:
        reasons.extend(fft_after.get('reasons', []))

    if delta.get('fft_high_frequency_ratio', 0) > 0.10:
        warnings_list.append('High-frequency energy increased unusually; possible synthetic texture/noise')
    if delta.get('fft_line_energy_ratio', 0) > 0.06:
        warnings_list.append('FFT line energy increased; possible repeated texture/grid/ringing')

    # Generative SR caution
    if MODEL_REGISTRY.get(settings.get('model'), {}).get('supports_generative_detail'):
        warnings_list.append('Generative SR may create new details; added detail is not automatically real detail')

    # Overall status
    if errors or 'FAIL' in statuses:
        status = 'FAIL'
    elif warnings_list or 'WARNING' in statuses:
        status = 'WARNING'
    else:
        status = 'PASS'

    if settings.get('stock_qc'):
        warnings_list.append('Stock QC: Technical QC only — platform acceptance is not guaranteed.')

    return {
        'status': status,
        'reasons': sorted(set(reasons)),
        'warnings': sorted(set(warnings_list)),
        'errors': sorted(set(errors)),
        'delta_metrics': delta,
    }



def save_pil_jpeg_flatten_white(src_path: str, dst_path: str, quality=95):
    """Save JPEG derivative. JPEG has no alpha, so transparent pixels are composited on white."""
    img = Image.open(src_path)
    img = ImageOps.exif_transpose(img)
    if img.mode in ('RGBA', 'LA') or ('transparency' in img.info):
        rgba = img.convert('RGBA')
        white = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
        img = Image.alpha_composite(white, rgba).convert('RGB')
        print(f'JPEG export: alpha flattened on white background -> {dst_path}')
    else:
        img = img.convert('RGB')
    quality = int(max(70, min(100, quality)))
    save_kwargs = {'quality': quality, 'optimize': True, 'subsampling': 0}
    img.save(dst_path, 'JPEG', **save_kwargs)
    return dst_path


def export_jpeg_derivatives(upscaled_path: str, qc_path: str, report: dict, compare_count: int):
    if not globals().get('EXPORT_JPEG', True):
        return {}
    quality = int(globals().get('JPEG_QUALITY', 95))
    stem = Path(upscaled_path).stem
    if compare_count > 1:
        up_jpg = OUTPUT_DIR / f'{stem}.jpg'
        qc_jpg = OUTPUT_DIR / f'{stem}_QC.jpg'
    else:
        up_jpg = OUTPUT_DIR / 'upscaled.jpg'
        qc_jpg = OUTPUT_DIR / 'upscaled_QC.jpg'
    save_pil_jpeg_flatten_white(upscaled_path, str(up_jpg), quality=quality)
    save_pil_jpeg_flatten_white(qc_path, str(qc_jpg), quality=quality)
    report['jpeg_outputs'] = {
        'upscaled_jpg': str(up_jpg),
        'upscaled_qc_jpg': str(qc_jpg),
        'quality': quality,
        'alpha_policy': 'JPEG has no alpha; transparent pixels are flattened on white if present.',
    }
    return report['jpeg_outputs']

def create_qc_panel_image(upscaled_path: str, qc_path: str, report: dict):
    img = Image.open(upscaled_path).convert('RGB')
    w, h = img.size
    panel_h = max(260, min(430, int(h * 0.16)))
    canvas = Image.new('RGB', (w, h + panel_h), (245, 245, 245))
    canvas.paste(img, (0, 0))
    draw = ImageDraw.Draw(canvas)
    try:
        font_title = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', max(18, w//60))
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', max(12, w//95))
    except Exception:
        font_title = font = ImageFont.load_default()
    y = h + 16
    status = report['quality_status']
    color = {'PASS': (20, 130, 60), 'WARNING': (210, 140, 0), 'FAIL': (190, 40, 40)}.get(status, (0,0,0))
    draw.rectangle([0, h, w, h + panel_h], fill=(245,245,245))
    draw.text((20, y), f"QC STATUS: {status}", fill=color, font=font_title)
    y += int(panel_h*0.18)
    def fmt_delta(x):
        return 'n/a' if x is None else f'{x:+}'
    lines = [
        f"Model: {report['model']} | Scale: {report['scale']}x | Resolution: {report['input_resolution'][0]}x{report['input_resolution'][1]} -> {report['output_resolution'][0]}x{report['output_resolution'][1]}",
        f"MUSIQ: {report['before_metrics'].get('MUSIQ')} -> {report['after_metrics'].get('MUSIQ')} ({fmt_delta(report['delta_metrics'].get('MUSIQ'))} delta)",
        f"NIQE: {report['before_metrics'].get('NIQE')} -> {report['after_metrics'].get('NIQE')} ({fmt_delta(report['delta_metrics'].get('NIQE'))} delta)",
        f"BRISQUE: {report['before_metrics'].get('BRISQUE')} -> {report['after_metrics'].get('BRISQUE')} ({fmt_delta(report['delta_metrics'].get('BRISQUE'))} delta)",
        f"Laplacian Variance: {report['before_metrics'].get('laplacian_variance')} -> {report['after_metrics'].get('laplacian_variance')} ({fmt_delta(report['delta_metrics'].get('laplacian_variance'))} delta)",
        f"FFT: {report['after_metrics'].get('fft',{}).get('status')} | {', '.join(report['after_metrics'].get('fft',{}).get('reasons',[])[:2])}",
        f"Processing: precision={report.get('precision')} tile={report['processing_information'].get('tile')} gpu={report.get('GPU')} elapsed={report['processing_information'].get('elapsed_sec')}s",
    ]
    for line in lines:
        draw.text((20, y), line[:220], fill=(20,20,20), font=font)
        y += int(panel_h*0.105)
    canvas.save(qc_path)
    return qc_path

QC_REPORTS = []
if not UPSCALE_RESULTS:
    print('No upscale results found. Run CELL 5 first.')
else:
    print('Computing BEFORE metrics once...')
    before_metrics = compute_quality_metrics(str(INPUT_IMAGE_PATH))
    print('Before metrics:', json.dumps(before_metrics, indent=2))
    for res in UPSCALE_RESULTS:
        if res.get('errors') and not res.get('output_path'):
            print(f"Skipping QC for failed model {res.get('model')}: {res.get('errors')}")
            continue
        print(f"\nComputing AFTER metrics for {res['model']}...")
        after_metrics = compute_quality_metrics(res['output_path'])
        interp = interpret_quality(before_metrics, after_metrics, res['settings'])
        report = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'input_filename': res['input_info']['filename'],
            'input_resolution': res['input_info']['resolution'],
            'output_resolution': res['output_resolution'],
            'model': res['model'],
            'scale': res['settings']['scale'],
            'model_settings': res['settings'],
            'GPU': HARDWARE.get('gpu'),
            'VRAM': HARDWARE.get('vram_total_gb'),
            'CUDA': HARDWARE.get('cuda'),
            'PyTorch': HARDWARE.get('pytorch'),
            'precision': res['processing_info'].get('effective_precision'),
            'before_metrics': before_metrics,
            'after_metrics': after_metrics,
            'delta_metrics': interp['delta_metrics'],
            'quality_status': interp['status'],
            'reasons': interp['reasons'],
            'warnings': list(sorted(set(interp['warnings'] + res.get('setting_warnings', [])))),
            'errors': interp['errors'] + res.get('errors', []),
            'processing_information': res['processing_info'],
            'export_jpeg': globals().get('EXPORT_JPEG', True),
            'jpeg_quality': int(globals().get('JPEG_QUALITY', 95)),
            'source_image_info': res['input_info'],
        }
        stem = Path(res['output_path']).stem
        report_path = OUTPUT_DIR / (f'quality_report_{res["model"].replace(" ","_").replace("/","_")}.json' if len(UPSCALE_RESULTS) > 1 else 'quality_report.json')
        qc_path = OUTPUT_DIR / (f'{stem}_QC.png' if len(UPSCALE_RESULTS) > 1 else 'upscaled_QC.png')
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        create_qc_panel_image(res['output_path'], str(qc_path), report)
        report['report_path'] = str(report_path)
        report['qc_image_path'] = str(qc_path)
        report['upscaled_path'] = res['output_path']
        export_jpeg_derivatives(res['output_path'], str(qc_path), report, compare_count=len(UPSCALE_RESULTS))
        # Rewrite report after JPEG paths are added.
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        QC_REPORTS.append(report)
        print(f"QC STATUS for {res['model']}: {report['quality_status']}")
        for r in report['reasons']: print('  ✅', r)
        for w in report['warnings']: print('  ⚠️', w)
        for e in report['errors']: print('  ❌', e)
    if not globals().get('KEEP_QC_CACHE', False):
        print(f'Clearing IQA metric cache ({len(IQA_CACHE)} metric model(s)).')
        IQA_CACHE.clear()
    else:
        print(f'KEEP_QC_CACHE=True; keeping {len(IQA_CACHE)} IQA metric model(s) cached.')
    clear_vram()


In [ ]:

# CELL 7 — Result Viewer
#@title CELL 7 — Result Viewer (Original / Upscaled / QC + Before/After Analysis)
if not QC_REPORTS:
    print('No QC reports found.')
else:
    for report in QC_REPORTS:
        display(Markdown(f"## Result: {report['model']} — {report['quality_status']}"))
        orig = Image.open(INPUT_IMAGE_PATH).convert('RGB')
        up = Image.open(report['upscaled_path']).convert('RGB')
        qc = Image.open(report['qc_image_path']).convert('RGB')
        # Display without modifying files
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(orig); axes[0].set_title(f"Original\n{orig.size[0]}x{orig.size[1]}"); axes[0].axis('off')
        axes[1].imshow(up); axes[1].set_title(f"Upscaled\n{up.size[0]}x{up.size[1]}"); axes[1].axis('off')
        axes[2].imshow(qc); axes[2].set_title('QC image with panel'); axes[2].axis('off')
        plt.tight_layout(); plt.show()

        rows = []
        for k in ['MUSIQ','NIQE','BRISQUE','laplacian_variance']:
            b = report['before_metrics'].get(k)
            a = report['after_metrics'].get(k)
            d = report['delta_metrics'].get(k)
            rows.append({'Metric': k, 'Before': b, 'After': a, 'Delta': d})
        rows.append({'Metric': 'FFT status', 'Before': report['before_metrics'].get('fft',{}).get('status'), 'After': report['after_metrics'].get('fft',{}).get('status'), 'Delta': report['delta_metrics'].get('fft_high_frequency_ratio')})
        show_table(rows, title='Before / After Metrics')

        display(Markdown('### Interpretation'))
        print('STATUS:', report['quality_status'])
        if report['reasons']:
            print('\nPASS signals / positive notes:')
            for x in report['reasons']: print('-', x)
        if report['warnings']:
            print('\nWarnings:')
            for x in report['warnings']: print('-', x)
        if report['errors']:
            print('\nErrors / fail signals:')
            for x in report['errors']: print('-', x)

    if len(QC_REPORTS) > 1:
        display(Markdown('## Compare Mode Summary'))
        rows = []
        for r in QC_REPORTS:
            rows.append({
                'Model': r['model'],
                'Resolution': f"{r['output_resolution'][0]}x{r['output_resolution'][1]}",
                'MUSIQ': r['after_metrics'].get('MUSIQ'),
                'NIQE': r['after_metrics'].get('NIQE'),
                'BRISQUE': r['after_metrics'].get('BRISQUE'),
                'Sharpness': r['after_metrics'].get('laplacian_variance'),
                'FFT': r['after_metrics'].get('fft',{}).get('status'),
                'Status': r['quality_status'],
            })
        show_table(rows, title='Compare Mode Summary')
        print('No automatic model recommendation is made. Please choose the model based on your own visual/QC requirements.')


In [ ]:

# CELL 8 — Download
#@title CELL 8 — Download results
CREATE_ZIP = True #@param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = False #@param {type:"boolean"}

import zipfile

if not QC_REPORTS:
    print('No files to download yet.')
else:
    files_to_zip = []
    for r in QC_REPORTS:
        for key in ['upscaled_path','qc_image_path','report_path']:
            p = Path(r[key])
            if p.exists():
                files_to_zip.append(p)
        for p_str in r.get('jpeg_outputs', {}).values():
            if isinstance(p_str, str):
                p = Path(p_str)
                if p.exists():
                    files_to_zip.append(p)
    # Preserve order but avoid duplicate paths.
    files_to_zip = list(dict.fromkeys(files_to_zip))
    if CREATE_ZIP:
        zip_path = OUTPUT_DIR / 'upscaled_result.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
            for p in files_to_zip:
                z.write(p, arcname=p.name)
        print('Created:', zip_path)
        if AUTO_DOWNLOAD_ZIP:
            files.download(str(zip_path))
    print('\nDownload files:')
    for p in files_to_zip:
        print('-', p)
    if CREATE_ZIP:
        print('-', zip_path)
    print('\nTo download manually in Colab: open the Files panel or run files.download(path).')


In [ ]:

# CELL 9 — Optional Self-Test / Smoke Checks
#@title CELL 9 — Optional Self-Test / Smoke Checks
RUN_SELF_TEST = False #@param {type:"boolean"}

if RUN_SELF_TEST:
    print('Running lightweight self-test with generated sample images...')
    test_dir = WORK_DIR / 'self_test'
    test_dir.mkdir(parents=True, exist_ok=True)

    # 1) RGB sample
    rgb = Image.new('RGB', (72, 64), (220, 60, 40))
    rgb_path = test_dir / 'sample_rgb.jpg'
    rgb.save(rgb_path, quality=95)

    # 2) Soft-alpha RGBA sample
    rgba = Image.new('RGBA', (64, 72), (20, 120, 240, 0))
    arr = np.asarray(rgba, dtype=np.uint8).copy()
    arr[..., 3] = np.tile(np.linspace(0, 255, arr.shape[1], dtype=np.uint8), (arr.shape[0], 1))
    rgba = Image.fromarray(arr, 'RGBA')
    rgba_path = test_dir / 'sample_soft_alpha.png'
    rgba.save(rgba_path)

    # 3) Palette transparency sample
    p_img = Image.new('P', (64, 64), 0)
    p_img.putpalette([255, 0, 0, 0, 255, 0] + [0, 0, 0] * 254)
    p_img.info['transparency'] = 0
    pal_path = test_dir / 'sample_palette_transparency.png'
    p_img.save(pal_path)

    # 4) CMYK sample
    cmyk = Image.new('CMYK', (64, 64), (0, 128, 128, 0))
    cmyk_path = test_dir / 'sample_cmyk.jpg'
    cmyk.save(cmyk_path)

    for path in [rgb_path, rgba_path, pal_path, cmyk_path]:
        img = Image.open(path)
        print(f'\nTesting {path.name}: mode={img.mode}, has_alpha={_has_alpha(img)}')
        cv_img, alpha = load_cv2_image_exif_safe(path, keep_alpha=True)
        assert cv_img.dtype == np.uint8 and cv_img.shape[2] == 3, 'model input must be uint8 BGR'
        if _has_alpha(img):
            assert alpha is not None and alpha.dtype == np.uint8, 'alpha should be cached as uint8'
            merged = merge_resized_alpha_if_needed(cv_img, alpha)
            assert merged.shape[2] == 4, 'alpha merge should produce BGRA'
        else:
            assert alpha is None, 'non-alpha sample should not return alpha'

    used = {}
    slugs = [safe_model_slug('A/B', used), safe_model_slug('A B', used), safe_model_slug('A_B', used)]
    print('Slug collision test:', slugs)
    assert len(set(slugs)) == len(slugs), 'slug collision handling failed'

    print('\n✅ Self-test passed. This does not download checkpoints or run SR models.')
else:
    print('RUN_SELF_TEST is False. Enable it to run lightweight generated-image checks.')
